In [1]:
# | eval: false

In [2]:
#| default_exp models

In [3]:
#| export
import torch
import torch.nn as nn


In [4]:
#| export
def f_net(ni,nf,nfs=(256,512,1024,512,256,128)):
  layers = []
  layers.append(nn.Linear(ni, nfs[0]))
  layers.append(nn.SiLU())
  for i in range(len(nfs) - 1):
    layers.append(nn.Linear(nfs[i], nfs[i+1]))
    layers.append(nn.SiLU())
  layers.append(nn.Linear(nfs[-1], nf))

  return nn.Sequential(*layers)

In [5]:
#| export
class FiLMLayer(nn.Module):
    def __init__(self, hidden_dim, cond_dim):
        super().__init__()
        self.scale = nn.Linear(cond_dim, hidden_dim)
        self.shift = nn.Linear(cond_dim, hidden_dim)
        nn.init.zeros_(self.scale.weight); nn.init.ones_(self.scale.bias)
        nn.init.zeros_(self.shift.weight); nn.init.zeros_(self.shift.bias)

    def forward(self, x, cond_emb):
        return x * (1 + torch.tanh(self.scale(cond_emb) - 1)) + self.shift(cond_emb)